# Chess Move Prediction Model Training
Train this notebook on Google Colab with GPU, then download the .pth file

## Dataset Format Expected
Your dataset should be a CSV with columns:
- `fen`: Board position in FEN notation
- `best_move`: Best move in UCI format (e.g., 'e2e4', 'e7e8q')
- `evaluation`: (Optional) Position evaluation score

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install torch torchvision python-chess pandas numpy tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import chess
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

## Configuration

In [ ]:
# Training configuration
CONFIG = {
    'dataset_path': 'your_dataset.csv',  # CHANGE THIS to your dataset path
    'batch_size': 256,
    'learning_rate': 0.001,
    'epochs': 50,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers': 4,
    'save_path': 'chess_model.pth',
    'val_split': 0.1,
}

print(f"Using device: {CONFIG['device']}")

## Board Encoding Functions

In [ ]:
def encode_board(fen):
    """
    Encode chess board from FEN to tensor representation
    Returns: (12, 8, 8) tensor
    12 channels: 6 piece types (P,N,B,R,Q,K) × 2 colors (white, black)
    """
    board = chess.Board(fen)
    tensor = np.zeros((12, 8, 8), dtype=np.float32)
    
    piece_idx = {
        chess.PAWN: 0,
        chess.KNIGHT: 1,
        chess.BISHOP: 2,
        chess.ROOK: 3,
        chess.QUEEN: 4,
        chess.KING: 5,
    }
    
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece:
            rank, file = divmod(square, 8)
            piece_type = piece_idx[piece.piece_type]
            color_offset = 0 if piece.color == chess.WHITE else 6
            tensor[piece_type + color_offset, rank, file] = 1
    
    return tensor

def encode_move(move_uci):
    """
    Encode UCI move to indices
    Returns: (from_square, to_square, promotion)
    from_square, to_square: 0-63
    promotion: 0=none, 1=knight, 2=bishop, 3=rook, 4=queen
    """
    move = chess.Move.from_uci(move_uci)
    from_square = move.from_square
    to_square = move.to_square
    
    promotion = 0
    if move.promotion:
        promotion_map = {
            chess.KNIGHT: 1,
            chess.BISHOP: 2,
            chess.ROOK: 3,
            chess.QUEEN: 4,
        }
        promotion = promotion_map.get(move.promotion, 0)
    
    return from_square, to_square, promotion

def decode_move(from_square, to_square, promotion_idx):
    """
    Decode move indices back to UCI format
    """
    promotion_map = {
        0: None,
        1: chess.KNIGHT,
        2: chess.BISHOP,
        3: chess.ROOK,
        4: chess.QUEEN,
    }
    
    move = chess.Move(from_square, to_square, promotion=promotion_map[promotion_idx])
    return move.uci()

## Dataset Class

In [ ]:
class ChessDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        """
        Args:
            csv_path: Path to CSV with columns: fen, best_move
        """
        self.data = pd.read_csv(csv_path)
        print(f"Loaded {len(self.data)} positions")
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Encode board
        board_tensor = encode_board(row['fen'])
        
        # Encode move
        from_sq, to_sq, promo = encode_move(row['best_move'])
        
        return {
            'board': torch.FloatTensor(board_tensor),
            'from_square': torch.LongTensor([from_sq]),
            'to_square': torch.LongTensor([to_sq]),
            'promotion': torch.LongTensor([promo]),
        }

## Model Architecture
CNN-based architecture inspired by AlphaZero

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        out = self.relu(out)
        return out

class ChessMovePredictor(nn.Module):
    def __init__(self, num_res_blocks=10, num_channels=256):
        super(ChessMovePredictor, self).__init__()
        
        # Initial convolution
        self.conv_input = nn.Conv2d(12, num_channels, kernel_size=3, padding=1)
        self.bn_input = nn.BatchNorm2d(num_channels)
        self.relu = nn.ReLU()
        
        # Residual tower
        self.res_blocks = nn.ModuleList([
            ResidualBlock(num_channels) for _ in range(num_res_blocks)
        ])
        
        # Policy head (move prediction)
        self.policy_conv = nn.Conv2d(num_channels, 32, kernel_size=1)
        self.policy_bn = nn.BatchNorm2d(32)
        self.policy_fc1 = nn.Linear(32 * 8 * 8, 512)
        
        # Three outputs: from_square, to_square, promotion
        self.from_square_head = nn.Linear(512, 64)
        self.to_square_head = nn.Linear(512, 64)
        self.promotion_head = nn.Linear(512, 5)  # none, N, B, R, Q
        
    def forward(self, x):
        # Input: (batch, 12, 8, 8)
        x = self.relu(self.bn_input(self.conv_input(x)))
        
        # Residual blocks
        for block in self.res_blocks:
            x = block(x)
        
        # Policy head
        policy = self.relu(self.policy_bn(self.policy_conv(x)))
        policy = policy.view(policy.size(0), -1)  # Flatten
        policy = self.relu(self.policy_fc1(policy))
        
        # Output heads
        from_square = self.from_square_head(policy)
        to_square = self.to_square_head(policy)
        promotion = self.promotion_head(policy)
        
        return from_square, to_square, promotion

## Training Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct_from = 0
    correct_to = 0
    correct_full = 0
    total = 0
    
    pbar = tqdm(dataloader, desc='Training')
    for batch in pbar:
        boards = batch['board'].to(device)
        from_squares = batch['from_square'].squeeze().to(device)
        to_squares = batch['to_square'].squeeze().to(device)
        promotions = batch['promotion'].squeeze().to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        from_pred, to_pred, promo_pred = model(boards)
        
        # Calculate loss
        loss_from = criterion(from_pred, from_squares)
        loss_to = criterion(to_pred, to_squares)
        loss_promo = criterion(promo_pred, promotions)
        
        loss = loss_from + loss_to + 0.1 * loss_promo  # Weight promotion less
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        _, from_predicted = torch.max(from_pred, 1)
        _, to_predicted = torch.max(to_pred, 1)
        _, promo_predicted = torch.max(promo_pred, 1)
        
        correct_from += (from_predicted == from_squares).sum().item()
        correct_to += (to_predicted == to_squares).sum().item()
        correct_full += ((from_predicted == from_squares) & 
                        (to_predicted == to_squares) & 
                        (promo_predicted == promotions)).sum().item()
        
        total += boards.size(0)
        total_loss += loss.item()
        
        pbar.set_postfix({
            'loss': total_loss / (pbar.n + 1),
            'acc_from': 100 * correct_from / total,
            'acc_to': 100 * correct_to / total,
            'acc_full': 100 * correct_full / total,
        })
    
    return total_loss / len(dataloader), correct_full / total

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct_full = 0
    total = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Validation'):
            boards = batch['board'].to(device)
            from_squares = batch['from_square'].squeeze().to(device)
            to_squares = batch['to_square'].squeeze().to(device)
            promotions = batch['promotion'].squeeze().to(device)
            
            # Forward pass
            from_pred, to_pred, promo_pred = model(boards)
            
            # Calculate loss
            loss_from = criterion(from_pred, from_squares)
            loss_to = criterion(to_pred, to_squares)
            loss_promo = criterion(promo_pred, promotions)
            loss = loss_from + loss_to + 0.1 * loss_promo
            
            # Calculate accuracy
            _, from_predicted = torch.max(from_pred, 1)
            _, to_predicted = torch.max(to_pred, 1)
            _, promo_predicted = torch.max(promo_pred, 1)
            
            correct_full += ((from_predicted == from_squares) & 
                            (to_predicted == to_squares) & 
                            (promo_predicted == promotions)).sum().item()
            
            total += boards.size(0)
            total_loss += loss.item()
    
    return total_loss / len(dataloader), correct_full / total

## Load Dataset

In [ ]:
# Load dataset
dataset = ChessDataset(CONFIG['dataset_path'])

# Split into train/val
val_size = int(len(dataset) * CONFIG['val_split'])
train_size = len(dataset) - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size]
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

## Initialize Model and Training

In [ ]:
# Initialize model
model = ChessMovePredictor(num_res_blocks=10, num_channels=256).to(CONFIG['device'])

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Starting training...")

## Training Loop

In [ ]:
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")
    print("-" * 50)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, CONFIG['device'])
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, CONFIG['device'])
    
    # Update scheduler
    scheduler.step(val_acc)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
        }, CONFIG['save_path'])
        print(f"✓ Saved best model with validation accuracy: {val_acc*100:.2f}%")

print("\nTraining complete!")
print(f"Best validation accuracy: {best_val_acc*100:.2f}%")

## Test Inference

In [ ]:
# Load best model
checkpoint = torch.load(CONFIG['save_path'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Test on a sample position
test_fen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"  # Starting position

board_tensor = torch.FloatTensor(encode_board(test_fen)).unsqueeze(0).to(CONFIG['device'])

with torch.no_grad():
    from_pred, to_pred, promo_pred = model(board_tensor)
    
    from_sq = torch.argmax(from_pred, dim=1).item()
    to_sq = torch.argmax(to_pred, dim=1).item()
    promo = torch.argmax(promo_pred, dim=1).item()
    
    predicted_move = decode_move(from_sq, to_sq, promo)
    
print(f"Test FEN: {test_fen}")
print(f"Predicted move: {predicted_move}")

# Verify it's a legal move
board = chess.Board(test_fen)
try:
    move = chess.Move.from_uci(predicted_move)
    if move in board.legal_moves:
        print("✓ Predicted move is legal!")
    else:
        print("✗ Predicted move is illegal")
except:
    print("✗ Invalid move format")

## Download Model
Download `chess_model.pth` from Colab and place it in the `models/` directory of your OpenCheck project

In [ ]:
# For Colab: Download the model file
from google.colab import files
files.download(CONFIG['save_path'])